In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS payment_gateway_catalog")
spark.sql("CREATE SCHEMA IF NOT EXISTS payment_gateway_catalog.raw")
spark.sql("CREATE SCHEMA IF NOT EXISTS payment_gateway_catalog.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS payment_gateway_catalog.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS payment_gateway_catalog.gold")

In [0]:
CATALOG_NAME = "payment_gateway_catalog"
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.raw.raw_payment_payloads (
    raw_payload STRING,
    _ingested_at TIMESTAMP
)
USING DELTA
""")
print(f"Catalog '{CATALOG_NAME}' and schemas successfully created")

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS payment_gateway_catalog.silver.silver_transactions (
        -- Unified fields
        transaction_id STRING,
        provider STRING,
        event_timestamp TIMESTAMP,
        original_amount DECIMAL(12, 2),
        original_currency STRING,
        fx_rate DECIMAL(8,4),
        amount_inr DECIMAL(12, 2),
        status STRING,
        is_success BOOLEAN,
        error_code STRING,
        error_description STRING,
        processed_at TIMESTAMP,
        event_date DATE,

        -- Razorpay Specific
        method STRING,
        vpa STRING,
        fee LONG,
        tax LONG,

        -- Slice Specific
        tenure_months LONG,
        merchant_category_code STRING,

        -- Dodo Payments Specific
        tax_jurisdiction STRING,
        vat_or_sales_tax_collected LONG
    )
    USING DELTA
    PARTITIONED BY (provider, event_date)
""")